In [ ]:
import pandas as pd
import torch
import time
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments

print("🚀 啟動階段四：SetFit 意圖解纏與模型微調 (黃金樣本強化版)")

# ==========================================
# 1. 載入三方資料：Snorkel, Normal, 與 Gold
# ==========================================

# A. 載入人工審核的黃金樣本 (強標籤：絕對正確)
df_gold = pd.read_csv('gold_candidates_reviewed.csv').dropna(subset=['text'])
# 對齊欄位名稱與標籤欄位
df_gold = df_gold.rename(columns={'proposed_category': 'snorkel_label'})

# B. 載入 Snorkel 自動標記資料 (弱標籤：量大但有噪聲)
df_snorkel = pd.read_csv('snorkel_labeled_training_data.csv')
df_snorkel = df_snorkel.rename(columns={'name': 'text'})

# C. 載入正常背景資料 (提供基準點)
df_normal = pd.read_csv('normal_pool_sample.csv')
df_normal = df_normal.rename(columns={'name': 'text'})
df_normal['snorkel_label'] = "正常困難負樣本"

# --- 關鍵：資料整合與去重 ---
# 如果 Snorkel 資料中包含了 Gold 已經人工標記過的句子，我們以 Gold 為準
df_snorkel_filtered = df_snorkel[~df_snorkel['text'].isin(df_gold['text'])].copy()

# 合併所有資料作為總訓練池
df_train_full = pd.concat([df_gold, df_snorkel_filtered, df_normal], ignore_index=True)

print("\n📊 [降採樣前] 混合訓練資料分佈：")
print(df_train_full['snorkel_label'].value_counts())

# ==========================================
# 2. 執行「精銳降採樣」策略
# ==========================================
MAX_SAMPLES_PER_CLASS = 400

# 我們確保人工標記的 Gold 樣本優先保留
def smart_sampling(group):
    # 如果樣本數本來就少於上限，全拿
    if len(group) <= MAX_SAMPLES_PER_CLASS:
        return group
    # 如果樣本數過多，則進行隨機抽樣
    return group.sample(n=MAX_SAMPLES_PER_CLASS, random_state=42)

df_train = df_train_full.groupby('snorkel_label', group_keys=False).apply(smart_sampling)
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n🎯 [降採樣後] SetFit 訓練分佈 (上限 {MAX_SAMPLES_PER_CLASS} 筆/類別)：")
print(df_train['snorkel_label'].value_counts())

# ------------------------------------------------
# 建立 Label 到 ID 的連續映射
unique_labels = sorted(df_train['snorkel_label'].unique())
label_to_id = {label: i for i, label in enumerate(unique_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(f"\n訓練類別映射表: {label_to_id}")

# 轉換為 Hugging Face Dataset 格式
df_train['label'] = df_train['snorkel_label'].map(label_to_id)
train_dataset = Dataset.from_pandas(df_train[['text', 'label']])

# ==========================================
# 3. 準備黃金驗證集 (作為期末考)
# ==========================================
# 我們依然使用人工審核的資料作為驗證集，確保模型真的學會了你的標記準則
eval_df = df_gold.copy()
eval_df['label'] = eval_df['snorkel_label'].map(label_to_id)
eval_dataset = Dataset.from_pandas(eval_df[['text', 'label']])

# ==========================================
# 4. 載入基礎模型與硬體加速
# ==========================================
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\n正在載入多語系基礎模型，使用加速裝置: {device}")

model = SetFitModel.from_pretrained(
    "paraphrase-multilingual-MiniLM-L12-v2",
    labels=list(label_to_id.keys())
)
model.to(device)

# ==========================================
# 5. 設定訓練參數與啟動 Trainer
# ==========================================
args = TrainingArguments(
    batch_size=64,                  
    num_epochs=3,                   
    evaluation_strategy="epoch",    
    save_strategy="epoch",
    load_best_model_at_end=True,    
    logging_steps=10,
    num_iterations=10               
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy"
)

print("\n🔥 開始進行對比學習微調 (含黃金樣本強化)...")
start_time = time.time()
trainer.train()

print(f"\n✅ 模型訓練完成！總耗時: {time.time() - start_time:.2f} 秒")

# ==========================================
# 6. 儲存模型與評估
# ==========================================
model_path = "./my_intent_classifier_setfit"
model.save_pretrained(model_path)
print(f"🎉 強化版模型已儲存至: {model_path}")

eval_results = trainer.evaluate()
print(f"\n📊 黃金樣本最終測試成績 (Accuracy): {eval_results['accuracy']:.4f}")

🚀 啟動階段四：SetFit 意圖解纏與模型微調 (精銳降採樣版)

📊 [降採樣前] 原始弱監督訓練分佈：
snorkel_label
借貸融資       2104
正常困難負樣本     508
黃色與特種行業     291
博弈          188
詐騙高風險招募      21
Name: count, dtype: int64

🎯 [降採樣後] 專為 SetFit 最佳化的精銳部隊 (上限 400 筆/類別)：
snorkel_label
正常困難負樣本    400
借貸融資       400
黃色與特種行業    291
博弈         188
詐騙高風險招募     21
Name: count, dtype: int64

訓練類別映射表: {'借貸融資': 0, '博弈': 1, '正常困難負樣本': 2, '詐騙高風險招募': 3, '黃色與特種行業': 4}

載入黃金樣本作為期末考驗證集...

正在載入多語系基礎模型，使用加速裝置: mps


/var/folders/_r/lwhdpchj7m5cftwpvm8_4bgw0000gn/T/ipykernel_39248/4017172889.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = df_train.groupby('snorkel_label', group_keys=False).apply(
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
The `evaluation_strategy` argument is deprecated and will be removed in a future version. Please use `eval_strategy` instead.


Map:   0%|          | 0/1300 [00:00<?, ? examples/s]


🔥 開始進行對比學習微調... (配對數量已受控，預計耗時將大幅縮短！)


***** Running training *****
  Num unique pairs = 26000
  Batch size = 64
  Num epochs = 3


Epoch,Training Loss,Validation Loss
1,0.004100,0.096547
2,0.000600,0.093543
3,0.000300,0.101970


The tokenizer you are loading from 'checkpoints/checkpoint-814' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



✅ 模型訓練完成！總耗時: 450.11 秒


***** Running evaluation *****


🎉 您專屬的意圖檢測模型已安全儲存至: ./my_intent_classifier_setfit

📊 黃金樣本驗證成績 (Accuracy): 0.7773


In [2]:
import yaml
import re

# ==========================================
# 0. 載入 YAML 地名設定 (確保與 Step 2 一致)
# ==========================================
def get_sanitizer(yaml_path='keywords.yaml'):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    loc_config = config.get('locations', {})
    all_locations = []
    for cat in loc_config:
        all_locations.extend(loc_config[cat])
    # 建立正則表達式
    pattern = re.compile(f"({'|'.join(all_locations)})")
    return pattern

city_pattern = get_sanitizer()

def preprocess_texts(texts, replacement="[LOC]"):
    if isinstance(texts, str):
        return city_pattern.sub(replacement, texts)
    return [city_pattern.sub(replacement, str(t)) if pd.notna(t) else "" for t in texts]

# ==========================================
# 6. 即時推論測試 (Inference Test - Sanitized Version)
# ==========================================
print("\n--- 🧠 實戰預測測試 (已整合地名脫敏) ---")

raw_test_sentences = [
    "日本東京灣的萬豪酒店住宿體驗分享",                # 正常困難樣本
    "全額貸，無勞保也可強力過件，當日放款",            # 借貸融資
    "群組受害者自救會，律師免費諮詢幫你拿回資金",       # 詐騙招募
    "註冊首存送 1000，今晚百家樂帶你飛",               # 博弈
    "誠徵台北外派公關，免經驗日領萬元",                 # 黃色特種 (帶有地名)
    "台南信貸，手續簡便當天撥款",                       # 之前最頭痛的地名偏見測試
    "八大"
]

# 核心步驟：推論前先進行脫敏處理
sanitized_sentences = preprocess_texts(raw_test_sentences)

# 讓模型預測 (傳入的是已經帶有 [LOC] 的文字)
preds = model.predict(sanitized_sentences)

# 為了方便觀察，我們同時印出清洗前後的差異
for raw, clean, label in zip(raw_test_sentences, sanitized_sentences, preds):
    print(f"原始: {raw}")
    if raw != clean:
        print(f"清洗: {clean}")
    print(f"👉 判定結果: [{label}]")
    print("-" * 30)


--- 🧠 實戰預測測試 (已整合地名脫敏) ---
原始: 日本東京灣的萬豪酒店住宿體驗分享
👉 判定結果: [黃色與特種行業]
------------------------------
原始: 全額貸，無勞保也可強力過件，當日放款
👉 判定結果: [借貸融資]
------------------------------
原始: 群組受害者自救會，律師免費諮詢幫你拿回資金
👉 判定結果: [正常困難負樣本]
------------------------------
原始: 註冊首存送 1000，今晚百家樂帶你飛
👉 判定結果: [正常困難負樣本]
------------------------------
原始: 誠徵台北外派公關，免經驗日領萬元
清洗: 誠徵[LOC]外派公關，免經驗日領萬元
👉 判定結果: [黃色與特種行業]
------------------------------
原始: 台南信貸，手續簡便當天撥款
清洗: [LOC]信貸，手續簡便當天撥款
👉 判定結果: [借貸融資]
------------------------------
原始: 八大
👉 判定結果: [正常困難負樣本]
------------------------------
